# Financial Document Intelligence — GPU steps

Runs the parts that need a GPU. Everything else runs on CPU in the repo.

1. Fine-tune LayoutLMv3 on CORD-v2 and write `artifacts/`
2. Index the corpus into Postgres + pgvector
3. Optionally serve extraction over a tunnel so the local API can call it

Runtime: **A100 High-RAM**.

In [ ]:
!git clone https://github.com/Perlious-Savage/financial-doc-intelligence.git
%cd financial-doc-intelligence

In [ ]:
# Kept minimal on purpose: Colab already ships torch, and every extra pinned
# package is another way the install can fail.
!pip install -q -r requirements-train.txt

## 1. Fine-tune

Smoke test first — confirms the whole path works before spending a real run on it.

In [ ]:
!python -m src.train_extractor --max-train 40 --epochs 1

Now the real run.

In [ ]:
!python -m src.train_extractor --epochs 8

In [ ]:
import json
print(json.dumps(json.load(open('artifacts/metrics.json')), indent=2))
print()
print(open('artifacts/per_field_report.txt').read())

### Commit the measured results back to the repo

The artifacts are the evidence. They belong in git, not only in this runtime.

In [ ]:
import os
os.environ['GH_TOKEN'] = ''  # paste a token with repo scope, or push from the local machine
!git config user.name "Perlious-Savage"
!git add -f artifacts/
!git commit -q -m "Add measured extraction results from CORD-v2 fine-tune"
# !git push

## 2. Index the corpus into pgvector

Embeddings need a model, so this runs GPU-side. Query time needs no model, which is why
the CPU service can serve similarity search on its own.

In [ ]:
!pip install -q -r requirements-index.txt

In [ ]:
import os
os.environ['DATABASE_URL'] = ''  # paste your Neon connection string

In [ ]:
import json
from datasets import load_dataset
from src.index import init_schema, embed_texts, upsert_documents, count_documents

init_schema()

ds = load_dataset('naver-clova-ix/cord-v2', split='test')

def receipt_text(record):
    gt = json.loads(record['ground_truth'])
    words = [w['text'] for line in gt.get('valid_line', []) for w in line.get('words', [])]
    return ' '.join(words)

texts = [receipt_text(r) for r in ds]
doc_ids = [f'cord-test-{i:04d}' for i in range(len(texts))]

print(f'embedding {len(texts)} documents ...')
vectors = embed_texts(texts)
upsert_documents(doc_ids, texts, vectors, [{'split': 'test'} for _ in texts])
print('indexed:', count_documents())

In [ ]:
# Sanity check: nearest neighbours of one receipt, excluding itself.
from src.index import find_similar
for hit in find_similar('cord-test-0000', k=5):
    print(round(hit['similarity'], 4), hit['doc_id'], hit['content'][:70])

## 3. Optional — serve extraction to the local API

Exposes `/predict` so the CPU service can run with `MODEL_BACKEND=remote`.

In [ ]:
!pip install -q pyngrok fastapi uvicorn nest_asyncio

In [ ]:
import io, nest_asyncio, uvicorn, torch
from fastapi import FastAPI, File, UploadFile
from PIL import Image
from transformers import AutoProcessor, LayoutLMv3ForTokenClassification

nest_asyncio.apply()
MODEL_DIR = 'outputs/layoutlmv3-cord'
processor = AutoProcessor.from_pretrained(MODEL_DIR, apply_ocr=True)
model = LayoutLMv3ForTokenClassification.from_pretrained(MODEL_DIR).eval().cuda()

serve = FastAPI()

@serve.post('/predict')
async def predict(file: UploadFile = File(...)):
    image = Image.open(io.BytesIO(await file.read())).convert('RGB')
    encoding = processor(image, truncation=True, padding='max_length',
                         max_length=512, return_tensors='pt')
    words = processor.tokenizer.batch_decode(encoding['input_ids'][0])
    with torch.no_grad():
        logits = model(**{k: v.cuda() for k, v in encoding.items()}).logits
    probabilities = logits.softmax(-1)[0]
    confidences, indices = probabilities.max(-1)
    tags = [model.config.id2label[int(i)] for i in indices]
    return {'words': words, 'tags': tags, 'confidences': [float(c) for c in confidences]}

In [ ]:
from pyngrok import ngrok
# ngrok.set_auth_token('YOUR_TOKEN')
tunnel = ngrok.connect(8000)
print('Set MODEL_ENDPOINT to:', tunnel.public_url)
uvicorn.run(serve, port=8000)